# 🚗 UA-DETRAC Trafik Veri Seti ile YOLO11 Araç Tespiti Eğitimi
### Bitirme Projesi - 45.000+ Gerçek Araç Etiketi ile Yüksek Doğruluklu Model Eğitimi

Bu notebook, **UA-DETRAC Benchmark Trafik Veri Seti (5.640 görsel, 45.000+ araç)** kullanılarak **YOLO11** modelini GPU hızlandırıcısı ile eğitmek için hazırlanmıştır.

**Kritik Adım:** Hızlı eğitim için Colab'in **T4 GPU** modunda olması gerekir:
👉 **Çalışma zamanı (Runtime) -> Çalışma zamanı türünü değiştir (Change runtime type) -> T4 GPU** seçip Kaydet'e basın.

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/aydin-eng/federated-traffic-learning/blob/main/traffic_vehicle_detection_yolo11.ipynb)


## 1. Donanım ve GPU Kontrolü
GPU'nun (T4) devrede olduğunu doğruluyoruz.


In [ ]:
import torch

if torch.cuda.is_available():
    print(f'✅ Harika! GPU devrede: {torch.cuda.get_device_name(0)}')
    !nvidia-smi
else:
    print('⚠️ UYARI: GPU devrede DEĞİL, CPU modundasınız!')
    print('Lütfen menüden: Çalışma zamanı -> Çalışma zamanı türünü değiştir -> T4 GPU seçip Kaydet deyin.')


## 2. Gerekli Kütüphanelerin Yüklenmesi
Ultralytics (YOLO11) kütüphanesini kuruyoruz.


In [ ]:
!pip install -q ultralytics


## 3. GitHub Deposunu Klonlama
Test fotoğrafları ve yardımcı araçları çekiyoruz.


In [ ]:
import os

if not os.path.exists('federated-traffic-learning'):
    !git clone https://github.com/aydin-eng/federated-traffic-learning.git
    %cd federated-traffic-learning
else:
    %cd federated-traffic-learning
    !git pull

print('Mevcut çalışma dizini:', os.getcwd())


## 4. UA-DETRAC Veri Setinin Yüklenmesi (`ua_detrac_yolo.zip`)
Hazırladığımız **`ua_detrac_yolo.zip` (374 MB)** dosyasını Colab'e bağlıyoruz.

**2 Kolay Yoldan Birini Seçin:**
* **Seçenek A (Google Drive - Önerilen):** 4.1 hücresini çalıştırın.
* **Seçenek B (Doğrudan Yükleme):** 4.2 hücresini çalıştırıp dosyayı seçin.


### 4.1 Seçenek A: Google Drive Bağlama (Önerilen)
`ua_detrac_yolo.zip` dosyasını Google Drive'ınızın ana sayfasına sürükleyip attıysanız bu hücreyi çalıştırın.


In [ ]:
import os
from google.colab import drive

drive.mount('/content/drive')

gdrive_zip = '/content/drive/MyDrive/ua_detrac_yolo.zip'
if os.path.exists(gdrive_zip):
    print('Google Drive\'daki arşiv açılıyor...')
    !unzip -q "{gdrive_zip}" -d /content/ua_detrac_dataset
    print('✅ Veri seti başarıyla /content/ua_detrac_dataset dizinine açıldı!')
else:
    print('Drive\'da ua_detrac_yolo.zip bulunamadı. Lütfen dosyanın Drive ana dizininde olduğundan emin olun.')


### 4.2 Seçenek B: Doğrudan Dosya Seçerek Yükleme
Google Drive yerine doğrudan bilgisayarınızdan yüklemek isterseniz bu hücreyi çalıştırıp `ua_detrac_yolo.zip` dosyasını seçin.


In [ ]:
import os
import shutil
from google.colab import files

# Eğer sol panelden elle /content/ altına sürüklendiyse doğrudan aç
if os.path.exists('/content/ua_detrac_yolo.zip'):
    print('/content/ua_detrac_yolo.zip bulundu, arşiv açılıyor...')
    !unzip -q /content/ua_detrac_yolo.zip -d /content/ua_detrac_dataset
    print('✅ Veri seti başarıyla açıldı!')
elif not os.path.exists('/content/ua_detrac_dataset/images/train'):
    print('Lütfen bilgisayarınızdan ua_detrac_yolo.zip dosyasını seçin:')
    uploaded = files.upload()
    for fn in uploaded.keys():
        if fn.endswith('.zip'):
            !unzip -q "{fn}" -d /content/ua_detrac_dataset
            print(f'✅ {fn} başarıyla açıldı!')
else:
    print('✅ Veri seti zaten yüklü ve hazır!')


## 5. data.yaml Konfigürasyonunu Doğrulama
Veri setinin dizinlerini ve 4 araç sınıfını (`car`, `motorcycle`, `bus`, `truck`) hazırlıyoruz.


In [ ]:
import os

yaml_content = """# UA-DETRAC YOLO Dataset Configuration
path: /content/ua_detrac_dataset
train: images/train
val: images/val
test: images/test

nc: 4
names:
  0: car
  1: motorcycle
  2: bus
  3: truck
"""

yaml_path = '/content/ua_detrac_dataset/data.yaml'
with open(yaml_path, 'w', encoding='utf-8') as f:
    f.write(yaml_content.strip() + '\n')

train_count = len(os.listdir('/content/ua_detrac_dataset/images/train')) if os.path.exists('/content/ua_detrac_dataset/images/train') else 0
val_count = len(os.listdir('/content/ua_detrac_dataset/images/val')) if os.path.exists('/content/ua_detrac_dataset/images/val') else 0
test_count = len(os.listdir('/content/ua_detrac_dataset/images/test')) if os.path.exists('/content/ua_detrac_dataset/images/test') else 0

print('✅ Veri Seti Doğrulandı:')
print(f' - Train Görselleri: {train_count}')
print(f' - Val Görselleri:   {val_count}')
print(f' - Test Görselleri:  {test_count}')
print(f' - data.yaml Yolu:   {yaml_path}')


## 6. YOLO11s Modelinin Başlatılması ve Eğitimi (Training) 🚀
45.000+ gerçek araç verisiyle **YOLO11s (Small - 9.4M Parametre)** modelini eğitiyoruz.

* **Model:** `yolo11s.pt` (Small)
* **Epochs:** 30
* **imgsz:** 640
* **batch:** 16 (GPU için optimize)


In [ ]:
import os
import torch
from ultralytics import YOLO

# Donanım kontrolü
device_arg = 0 if torch.cuda.is_available() else 'cpu'
print(f'Kullanılan donanım: {device_arg} (CUDA: {torch.cuda.is_available()})')

# Önceden eğitilmiş YOLO11s modelini yükle
model = YOLO('yolo11s.pt')
print('YOLO11s modeli yüklendi. UA-DETRAC üzerinde eğitim başlatılıyor...')

results = model.train(
    data='/content/ua_detrac_dataset/data.yaml',
    epochs=30,
    imgsz=640,
    batch=16,
    device=device_arg,          # Otomatik GPU/CPU seçimi
    project='runs/train',
    name='yolo11s_uad_exp',
    patience=10,
    save=True,
    plots=True,
    seed=42,
    verbose=True
)


## 7. Modelin Test Kümesinde Değerlendirilmesi (Evaluation)
Eğitilen en iyi ağırlık (`best.pt`) ile 1.594 adetlik bağımsız test kümesi üzerinde başarıyı ölçüyoruz.


In [ ]:
import os
from ultralytics import YOLO

# Test kümesi değerlendirmesi
metrics = model.val(split='test')

print('\n' + '=' * 50)
print('   UA-DETRAC TEST KÜMESİ BAŞARI SONUÇLARI')
print('=' * 50)
print(f"mAP@50:     {metrics.box.map50:.4f} (%{metrics.box.map50 * 100:.2f})")
print(f"mAP@50-95:  {metrics.box.map:.4f} (%{metrics.box.map * 100:.2f})")
print(f"Precision:  {metrics.box.mp:.4f} (%{metrics.box.mp * 100:.2f})")
print(f"Recall:     {metrics.box.mr:.4f} (%{metrics.box.mr * 100:.2f})")
print('=' * 50)


## 8. Eğitim Grafikleri ve Karışıklık Matrisi
Kayıp grafikleri (`results.png`), Precision-Recall eğrisi ve Confusion Matrix sonuçlarını görüntülüyoruz.


In [ ]:
import os
import glob
from IPython.display import Image, display

plots = [
    'runs/train/yolo11s_uad_exp/results.png',
    'runs/train/yolo11s_uad_exp/confusion_matrix_normalized.png',
    'runs/train/yolo11s_uad_exp/BoxPR_curve.png'
]

for p in plots:
    if os.path.exists(p):
        print(f'Görsel: {os.path.basename(p)}')
        display(Image(filename=p, width=700))


## 9. Gerçek Trafik Test Fotoğrafı Üzerinde Test (Inference)
Özel trafik fotoğrafımız (`test_samples/user_test_photo.jpg`) üzerinde çıkarım yapıyoruz.


In [ ]:
import os
import glob
from IPython.display import Image, display

custom_photo = 'test_samples/user_test_photo.jpg'
if os.path.exists(custom_photo):
    custom_pred = model.predict(
        source=custom_photo,
        conf=0.25,
        save=True,
        project='runs/detect',
        name='custom_test_result'
    )
    pred_imgs = glob.glob('runs/detect/custom_test_result/*.jpg')
    if pred_imgs:
        print('Gerçek trafik test fotoğrafı üzerindeki tespitler:')
        display(Image(filename=pred_imgs[0], width=700))
else:
    print('test_samples/user_test_photo.jpg bulunamadı.')


## 10. Sonuçların Otomatik Paketlenmesi ve İndirilmesi 📦
Eğitilen `best.pt` ağırlık dosyasını ve tüm grafikleri tek bir `yolo11_results.zip` dosyası haline getirip indirir.


In [ ]:
import os
import shutil
from google.colab import files

exp_dir = 'runs/train/yolo11s_uad_exp'
best_weights = f'{exp_dir}/weights/best.pt'
if os.path.exists(best_weights):
    size_mb = os.path.getsize(best_weights) / (1024 * 1024)
    print(f'✅ Yeni model ağırlığı hazır: {best_weights} ({size_mb:.2f} MB)')

print('📦 Sonuçlar paketleniyor ve yolo11_results.zip indiriliyor...')
shutil.make_archive('yolo11_results', 'zip', exp_dir)
files.download('yolo11_results.zip')
print('🚀 yolo11_results.zip indirme başlatıldı!')
